# ForecastLLM - Week 8 Results
## From Pricing Agents to Forecast Alert Workflows


This project follows Ed Donner's capstone structure and adapts the domain from Amazon/product pricing to forecasting.

Weeks 6-7 keep M4 hourly data to preserve forecasting discipline and model-evaluation rigor.

Week 8 pivots to weekly gasoline prices because they align well with the course's price/action/agent pedagogy.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ALERT_THRESHOLD = 0.05
EXPECTED_SCHEMA_VERSION = "forecastllm.week8.notification.v1"


In [2]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists()), cwd)
RUN_DIR = PROJECT_ROOT / "week8" / "run"

DAY4_SUMMARY_PATH = RUN_DIR / "week8_day4_summary.json"
DAY4_NOTIFICATIONS_PATH = RUN_DIR / "week8_day4_notifications.jsonl"
DAY5_REPORT_PATH = RUN_DIR / "week8_day5_report.md"

missing = [str(p) for p in [DAY4_SUMMARY_PATH, DAY4_NOTIFICATIONS_PATH, DAY5_REPORT_PATH] if not p.exists()]
if missing:
    missing_text = "\n".join(missing)
    raise FileNotFoundError(
        "Missing required Week 8 artifacts. Run week8/day4.ipynb and week8/day5.ipynb first.\n"
        f"Missing paths:\n{missing_text}"
    )

print(f"Loaded artifacts from: {RUN_DIR}")


Loaded artifacts from: /home/geo/Projects/Python/forecastllm/week8/run


In [3]:
with DAY4_SUMMARY_PATH.open("r", encoding="utf-8") as f:
    day4_summary = json.load(f)

notifications_df = pd.read_json(DAY4_NOTIFICATIONS_PATH, lines=True)
if notifications_df.empty:
    raise RuntimeError("Day 4 notifications artifact exists but is empty.")

if "schema_version" not in notifications_df.columns:
    raise ValueError(
        "Missing 'schema_version' in notifications artifact. Re-run week8/day4.ipynb."
    )

schema_values = sorted(set(notifications_df["schema_version"].dropna().astype(str)))
schema_version_display = schema_values[0] if len(schema_values) == 1 else schema_values

display_summary = {
    "total_rows": int(day4_summary.get("total_rows", 0)),
    "alert_count": int(day4_summary.get("num_alerts", 0)),
    "increase_alerts": int(day4_summary.get("num_increase", 0)),
    "decrease_alerts": int(day4_summary.get("num_decrease", 0)),
    "threshold": ALERT_THRESHOLD,
    "schema_version": schema_version_display,
}

print("Day 4 Summary")
print(json.dumps(display_summary, indent=2))

if schema_version_display != EXPECTED_SCHEMA_VERSION:
    print(f"Warning: expected schema_version={EXPECTED_SCHEMA_VERSION}, got {schema_version_display}")


Day 4 Summary
{
  "total_rows": 271,
  "alert_count": 238,
  "increase_alerts": 128,
  "decrease_alerts": 110,
  "threshold": 0.05,
  "schema_version": "forecastllm.week8.notification.v1"
}


In [4]:
notifications_df["metadata"] = notifications_df["metadata"].apply(lambda x: x if isinstance(x, dict) else {})
notifications_df["timestamp"] = notifications_df["metadata"].apply(lambda m: m.get("cutoff_timestamp"))
notifications_df["alert_type"] = notifications_df["metadata"].apply(lambda m: m.get("alert_type"))
notifications_df["forecast_change"] = notifications_df["metadata"].apply(lambda m: m.get("change"))
notifications_df["baseline_used"] = notifications_df["metadata"].apply(lambda m: m.get("baseline_used"))

review_table = notifications_df[
    ["timestamp", "alert_type", "subject", "forecast_change", "baseline_used"]
].copy()

review_table.head(15)


,timestamp,alert_type,subject,forecast_change,baseline_used
0,2021-03-01,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,-0.210,lag_52
1,2021-03-08,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,-0.336,lag_52
2,2021-03-15,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,-0.523,lag_52
3,2021-03-22,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,-0.733,lag_52
4,2021-03-29,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,-0.860,lag_52
5,2021-04-05,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,-0.928,lag_52
6,2021-04-12,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,-1.004,lag_52
7,2021-04-19,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,-1.037,lag_52
8,2021-04-26,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,-1.082,lag_52
9,2021-05-03,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,-1.083,lag_52


In [5]:
report_md = DAY5_REPORT_PATH.read_text(encoding="utf-8")
display(Markdown(report_md))


# Week 8 Day 5 Forecast Alert Review

## Headline Summary
- Total rows evaluated: 271
- Alerts generated: 238
- Increase alerts: 128
- Decrease alerts: 110
- Fixed alert threshold: 0.05
- Notification schema version: forecastllm.week8.notification.v1

## Top 3 Alerts (by absolute forecast change)
1. [decrease_alert] [GASREGW] decrease_alert: forecast decrease of $1.946 on 2022-06-20 | change=-1.9460000000000002 | baseline=lag_52
2. [decrease_alert] [GASREGW] decrease_alert: forecast decrease of $1.871 on 2022-06-27 | change=-1.870999999999999 | baseline=lag_52
3. [decrease_alert] [GASREGW] decrease_alert: forecast decrease of $1.807 on 2022-06-13 | change=-1.807 | baseline=lag_52

## Sample Notification Body
```text
ForecastLLM Gasoline Alert

Series/commodity: GASREGW (U.S. Regular All Formulations Gasoline Price (Weekly))
Timestamp: 2021-03-01
Current/last observed price: $2.633 dollars_per_gallon
Forecast price: $2.423 dollars_per_gallon
Forecast change: $-0.210 dollars_per_gallon
Alert type: decrease_alert
Baseline used: lag_52

Explanation: Baseline 'lag_52' indicates a decrease of $0.210 per gallon relative to the last observed price, exceeding threshold for analyst review.

Email draft only. No email has been sent.
```

## Next-Step TODOs
- Confirm recipients and routing policy by alert type.
- Add delivery status tracking once sending is introduced.
- Define how this report rolls into week8/results.ipynb.

## Fixed Alert Threshold

The workflow uses a fixed threshold of **$0.05 per gallon**.

This is intentionally simple and human-interpretable, so alert behavior is easy to explain in an operations context.

It is not presented as a statistically optimized threshold.


## What This Preserves from Donner's Original Project

- Structured progression across days and system layers.
- Agents/services/workflow mindset, even when implemented locally.
- Opportunity/alert detection as the core decision trigger.
- Notification/reporting as the user-facing action layer.
- Human-reviewable outputs as the final quality gate.


## What Changed

- Pricing -> forecasting.
- Product/deal -> forecast case/alert.
- Pushover notification -> email-ready notification records.
- Modal/deployment -> local deterministic workflow.
- Synthetic/toy data -> real public gasoline series for Week 8.


## Limitations

- Single gasoline series.
- Simple baseline forecasts.
- Fixed threshold.
- No uncertainty modeling.
- No real email sending.
- No live agents yet.
- Run artifacts are local and not committed.


## Future Work

- Add regional gasoline series.
- Add dynamic thresholds.
- Add LLM-generated explanations.
- Add Gmail API integration.
- Add Scribe/Clerk notification workflow integration.
- Add optional deployed service later.
